In [2]:
# DASK client set

import os
import sys
from dask.distributed import Client
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json', threads_per_worker=2, n_workers=6)
client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json')
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler_10.json')  

# add private module path for workers
# client.run(lambda: os.environ.update({'PYTHONPATH': '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'}))
# def add_path():
#     if '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2' not in sys.path:
#         sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')

# client.run(add_path)

def setup_module_path():
    module_path = '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'
    if module_path not in sys.path:
        sys.path.append(module_path)

client.run(setup_module_path)

client

# get path for path changes in Jupyter notebook: File - Open from Path - insert relative_path
notebook_path = os.path.abspath(".")
_, _, relative_path = notebook_path.partition('/all/')
relative_path = '/all/' + relative_path
relative_path

'/all/Model/CESM2/Earth_System_Predictability/DIC'

# Load modules

In [3]:
# load public modules

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.ticker as mticker
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from scipy.interpolate import griddata
import cmocean
from cmcrameri import cm
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import cftime
import pop_tools
from pprint import pprint
import time
import subprocess
import re as re_mod
import cftime
import datetime
from scipy.stats import ttest_1samp


In [4]:
import xcesm
import gsw

In [5]:
# load private modules

import sys
sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')
from KYY_CESM2_NWP_preprocessing import CESM2_NWP_config
# import KYY_CESM2_preprocessing
# import importlib
# importlib.reload(KYY_CESM2_preprocessing)

# Variable configuration

In [6]:

cfgs = {}
# varlist = ["DIC", "TEMP", "SALT"]
# varlist = ["DIC_ALT_CO2"]
varlist = ["DIC", "SALT"]

for varname in varlist:
    cfg = CESM2_NWP_config()
    cfg.year_s = 1954
    cfg.year_e = 2020
    cfg.setvar(varname)
    cfgs[varname] = cfg

if cfgs[varname].comp=='ocn':
    ds_grid = pop_tools.get_grid('POP_gx1v7')

# Read dataset

In [7]:
ds_grid.dz
ds_grid.TAREA

<xarray.DataArray 'TAREA' (nlat: 384, nlon: 320)> Size: 983kB
array([[1.12478609e+13, 1.12464644e+13, 1.12436015e+13, ...,
        1.12436015e+13, 1.12464644e+13, 1.12478609e+13],
       [1.45745047e+13, 1.45745047e+13, 1.45745047e+13, ...,
        1.45745047e+13, 1.45745047e+13, 1.45745047e+13],
       [1.52530781e+13, 1.52530781e+13, 1.52530781e+13, ...,
        1.52530781e+13, 1.52530781e+13, 1.52530781e+13],
       ...,
       [8.81450970e+12, 8.81311214e+12, 8.81016095e+12, ...,
        8.81016095e+12, 8.81311214e+12, 8.81450970e+12],
       [8.13997430e+12, 8.13851312e+12, 8.13544719e+12, ...,
        8.13544719e+12, 8.13851312e+12, 8.13997430e+12],
       [7.43222977e+12, 7.43072723e+12, 7.42759165e+12, ...,
        7.42759165e+12, 7.43072723e+12, 7.43222977e+12]])
Dimensions without coordinates: nlat, nlon
Attributes:
    units:        cm^2
    long_name:    area of T cells
    coordinates:  TLONG TLAT

In [8]:
# lat_range = slice(23, 38)
# lon_range = slice(120, 180)
lat_range = slice(19, 40)
lon_range = slice(120, 180)

In [46]:
# define preprocessing function

# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]
# exceptcv=['time','lon','lat','lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'dz', 'z_t_2', cfg_var_DIC.var, cfg_var_TEMP.var]


# exceptcv = [
#     'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
# ] + [cfg.var for cfg in cfgs.values()]
exceptcv = [
    'time', 'lon', 'lat', 'lev', 'TAREA', 'TLONG', 'TLAT', 'z_t', 'MTD'
] + [cfg.var for cfg in cfgs.values()]


def process_coords_3d(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        # ds = ds.isel(z_t=slice(0, 39)) # ~39 layer (1000m)
        # ds = (ds.isel(z_t=slice(1, 39)) * ds.dz).sum(dim='z_t') / ds.dz.sum(dim='z_t')
        return ds
    else:
        return ds.set_coords(coord_vars)



def process_coords_3d_LE(ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    """
    Preprocessor function for CESM POP-style datasets.
    - Normalizes vertical coordinate: if z_t or z_t_2 exists, rename to 'depth'.
    - Replaces its values with z_t_new for consistency.
    - Optionally drops unnecessary coordinate variables for faster concatenation.
    """
    z_t_new = np.array([5.0000000e+00, 1.5000000e+01, 2.5000000e+01, 3.5000000e+01,
       4.5000000e+01, 5.5000000e+01, 6.5000000e+01, 7.5000000e+01,
       8.5000000e+01, 9.5000000e+01, 1.0500000e+02, 1.1500000e+02,
       1.2500000e+02, 1.3500000e+02, 1.4500000e+02, 1.5500000e+02,
       1.6509839e+02, 1.7547903e+02, 1.8629126e+02, 1.9766026e+02,
       2.0971138e+02, 2.2257828e+02, 2.3640883e+02, 2.5137015e+02,
       2.6765421e+02, 2.8548364e+02, 3.0511920e+02, 3.2686798e+02,
       3.5109348e+02, 3.7822760e+02, 4.0878464e+02, 4.4337769e+02,
       4.8273669e+02, 5.2772797e+02, 5.7937286e+02, 6.3886261e+02,
       7.0756329e+02, 7.8700250e+02, 8.7882520e+02, 9.8470581e+02,
       1.1062042e+03, 1.2445669e+03, 1.4004972e+03, 1.5739464e+03,
       1.7640033e+03, 1.9689442e+03, 2.1864565e+03, 2.4139714e+03,
       2.6490012e+03, 2.8893845e+03, 3.1334045e+03, 3.3797935e+03,
       3.6276702e+03, 3.8764519e+03, 4.1257681e+03, 4.3753926e+03,
       4.6251904e+03, 4.8750835e+03, 5.1250278e+03, 5.3750000e+03])
    
    # ------------------------------------------------------
    # 1️⃣ Normalize vertical coordinate name
    # ------------------------------------------------------
    if "z_t_2" in ds.dims:
        ds = ds.rename({"z_t_2": "depth"})
    elif "z_t" in ds.dims:
        ds = ds.rename({"z_t": "depth"})
    else:
        print("[Warning] No vertical coordinate (z_t or z_t_2) found — skipped.")
        return ds

    # Drop any leftover z_t/z_t_2 coordinate variable if it exists
    ds = ds.drop_vars(["z_t", "z_t_2"], errors="ignore")

    # ------------------------------------------------------
    # 2️⃣ Replace coordinate values with z_t_new
    # ------------------------------------------------------
    if "depth" in ds.coords:
        if len(ds["depth"]) == len(z_t_new):
            ds = ds.assign_coords(depth=z_t_new)
        else:
            print(f"[Warning] depth length mismatch: {len(ds['depth'])} vs {len(z_t_new)}")
    else:
        print("[Warning] depth coordinate missing after renaming.")

    # ------------------------------------------------------
    # 3️⃣ Clean up coordinate references inside variable attributes
    # ------------------------------------------------------
    for v in ds.data_vars:
        if "coordinates" in ds[v].attrs:
            ds[v].attrs["coordinates"] = (
                ds[v].attrs["coordinates"]
                .replace("z_t_2", "depth")
                .replace("z_t", "depth")
            )

    # ------------------------------------------------------
    # 4️⃣ Drop unnecessary coordinate variables and slice time
    # ------------------------------------------------------
    coord_vars = []
    for v in np.array(ds.coords):
        if v not in except_coord_vars:
            coord_vars.append(v)
    for v in np.array(ds.data_vars):
        if v not in except_coord_vars:
            coord_vars.append(v)

    if drop:
        ds = ds.drop(coord_vars, errors="ignore")
        ds = ds.sel(time=slice(sd, ed))
    else:
        ds = ds.set_coords(coord_vars)

    return ds


start_date = cftime.DatetimeNoLeap(cfgs[varname].year_s, 2, 1)
end_date = cftime.DatetimeNoLeap(cfgs[varname].year_e+1, 1, 1)


# ds = ds.isel(lev=slice(1, 11))

# ODA part

In [26]:
import os
import glob
import re
import time
import xarray as xr
from collections import defaultdict

BASE_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/regridded"

vars = varlist
results = {}
file_list_assm = {}
file_list_le = {}

for var in vars:

    print(f"\n===== Collecting regridded files for {var} =====")

    cfg_var = cfgs[var]
    cfg_var.var = var

    # ============================================================
    # ======================= ODA (ORIGINAL) =====================
    # ============================================================
    BASE_RGD = os.path.join(BASE_ROOT, "ODA", "NWP", "ocn")

    var_dir = os.path.join(BASE_RGD, var)

    if os.path.isdir(var_dir):

        ens_dirs = sorted(
            [
                d for d in os.listdir(var_dir)
                if os.path.isdir(os.path.join(var_dir, d))
            ]
        )

        rgd_file_list = []

        for ens in ens_dirs:
            ens_path = os.path.join(var_dir, ens)
            files = sorted(
                glob.glob(os.path.join(ens_path, "regridded_*.nc"))
            )
            rgd_file_list.append(files)

        file_list_assm[var] = rgd_file_list

        print(f"\n===== Processing ODA {var} =====")

        t0 = time.time()
        cfg_var.ODA_path_load(cfg_var.var)
        cfg_var.ODA_ds_rgd = xr.open_mfdataset(
            file_list_assm[var][10:20],
            chunks={'time': 12},
            combine='nested',
            concat_dim=[[*cfg_var.ODA_ensembles][10:20], 'time'],
            parallel=True,
            preprocess=lambda ds: process_coords_3d(ds, start_date, end_date),
            decode_cf=True,
            decode_times=True,
        )
        cfg_var.ODA_ds_rgd = cfg_var.ODA_ds_rgd.rename({"concat_dim": "ens_ODA"})
        cfg_var.ODA_ds_rgd = cfg_var.ODA_ds_rgd.sortby("time")
        print(f"  ODA read: {time.time() - t0:.1f} s")

    # ============================================================
    # ======================= WDA ================================
    # ============================================================
    BASE_WDA = os.path.join(BASE_ROOT, "WDA", "NWP", "ocn", var)

    if os.path.isdir(BASE_WDA):

        print(f"\n===== Processing WDA {var} =====")

        files = sorted(glob.glob(os.path.join(BASE_WDA, "regridded_*.nc")))

        t0 = time.time()
        cfg_var.WDA_ds_rgd = xr.open_mfdataset(
            files,
            chunks={'time': 12},
            combine='by_coords',
            parallel=True,
            preprocess=lambda ds: process_coords_3d(ds, start_date, end_date),
            decode_cf=True,
            decode_times=True,
        )
        cfg_var.WDA_ds_rgd = cfg_var.WDA_ds_rgd.sortby("time")
        print(f"  WDA read: {time.time() - t0:.1f} s")

    # ============================================================
    # ======================= LE (ORIGINAL STYLE) ================
    # ============================================================
    BASE_LE = os.path.join(BASE_ROOT, "LENS2", "NWP", "ocn", var)

    if os.path.isdir(BASE_LE):

        print(f"\n===== Processing LE {var} =====")

        ens_dirs = [
            d for d in os.listdir(BASE_LE)
            if os.path.isdir(os.path.join(BASE_LE, d))
        ]

        grouped = defaultdict(list)

        for ens_dir in ens_dirs:
            match = re.search(r"(LE2-[0-9]+\.[0-9]+)", ens_dir)
            if match:
                member_id = match.group(1)
                full_path = os.path.join(BASE_LE, ens_dir)

                files = glob.glob(
                    os.path.join(full_path, "regridded_*.nc")
                )

                grouped[member_id].extend(files)

        final_file_list = []

        for member in sorted(grouped.keys()):
            files_sorted = sorted(grouped[member])
            final_file_list.append(files_sorted)

        file_list_le[var] = final_file_list

        t0 = time.time()
        cfg_var.LE_ds_rgd = xr.open_mfdataset(
            file_list_le[var],
            chunks={'time': 12},
            combine='nested',
            concat_dim=[[*sorted(grouped.keys())], 'time'],
            parallel=True,
            preprocess=lambda ds: process_coords_3d(ds, start_date, end_date),
            decode_cf=True,
            decode_times=True,
        )
        cfg_var.LE_ds_rgd = cfg_var.LE_ds_rgd.rename({"concat_dim": "ens_LE"})
        cfg_var.LE_ds_rgd = cfg_var.LE_ds_rgd.sortby("time")
        print(f"  LE read: {time.time() - t0:.1f} s")

    results[var] = cfg_var


===== Collecting regridded files for DIC =====

===== Processing ODA DIC =====
  ODA read: 2.7 s

===== Processing WDA DIC =====
  WDA read: 0.2 s

===== Processing LE DIC =====
  LE read: 3.9 s

===== Collecting regridded files for SALT =====

===== Processing ODA SALT =====
  ODA read: 3.2 s

===== Processing WDA SALT =====
  WDA read: 0.2 s

===== Processing LE SALT =====
  LE read: 4.2 s


In [47]:
BASE_RGD = "/mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/ODA/NWP/ocn"

var = 'MTD'
print(f"\n===== Collecting regridded files for {var} =====")

var_dir = os.path.join(BASE_RGD, var)

# Detect ensemble directories automatically
ens_dirs = sorted(
    [
        d for d in os.listdir(var_dir)
        if os.path.isdir(os.path.join(var_dir, d))
    ]
)

print(f"Detected ensembles: {ens_dirs}")

rgd_file_list = []

for ens in ens_dirs:
    ens_path = os.path.join(var_dir, ens)

    # Collect all regridded files regardless of time segmentation
    files = sorted(
        glob.glob(os.path.join(ens_path, "regridded_*.nc"))
    )
    rgd_file_list.append(files)

file_list_assm[var] = rgd_file_list

var = 'MTD'

cfg = CESM2_NWP_config()
cfg.year_s = 1954
cfg.year_e = 2020
# cfg.setvar(var)
cfgs[var] = cfg

print(f"\n===== Processing {var} =====")

cfg_var = cfgs[var]
cfg_var.var = var

# ================= ODA =================
t0 = time.time()
cfg_var.ODA_ds_rgd = xr.open_mfdataset(
    file_list_assm[var][10:20],
    chunks={'time': 12},
    combine='nested',
    concat_dim=[[*cfgs['DIC'].ODA_ensembles][10:20], 'time'],
    parallel=True,
    preprocess=lambda ds: process_coords_3d(ds, start_date, end_date),
    decode_cf=True,
    decode_times=True,
)
cfg_var.ODA_ds_rgd = cfg_var.ODA_ds_rgd.rename({"concat_dim": "ens_ODA"})
cfg_var.ODA_ds_rgd = cfg_var.ODA_ds_rgd.sortby("time")
print(f"  ODA read: {time.time() - t0:.1f} s")

results[var] = cfg_var



import os
import glob

BASE_RGD = "/mnt/lustre/proj/kimyy/tr_sysong/fld/regridded/WDA/NWP/ocn"

var = "MTD"

print(f"\n===== Collecting WDA regridded files for {var} =====")

var_dir = os.path.join(BASE_RGD, var)

if not os.path.isdir(var_dir):
    raise FileNotFoundError(f"WDA directory not found: {var_dir}")

# WDA: ens 폴더 없음 → 바로 파일 수집
files = sorted(
    glob.glob(os.path.join(var_dir, "regridded_*.nc"))
)

print(f"WDA files found: {len(files)}")

# ODA와 동일 인터페이스 유지 (list of list)
file_list_WDA = {}
file_list_WDA[var] = [files]


# ================= WDA MTD =================
import time

t0 = time.time()

cfg_var.WDA_ds_rgd = xr.open_mfdataset(
    file_list_WDA[var][0],   # ← 핵심 수정
    chunks={'time': 12},
    combine='nested',
    concat_dim='time',
    parallel=True,
    preprocess=lambda ds: process_coords_3d(ds, start_date, end_date),
    decode_cf=True,
    decode_times=True,
)

cfg_var.WDA_ds_rgd = cfg_var.WDA_ds_rgd.sortby("time")

print(f"  WDA read: {time.time() - t0:.1f} s")



===== Collecting regridded files for MTD =====
Detected ensembles: ['en4.2_ba-10p1', 'en4.2_ba-10p2', 'en4.2_ba-10p3', 'en4.2_ba-10p4', 'en4.2_ba-10p5', 'en4.2_ba-20p1', 'en4.2_ba-20p2', 'en4.2_ba-20p3', 'en4.2_ba-20p4', 'en4.2_ba-20p5', 'projdv7.3_ba-10p1', 'projdv7.3_ba-10p2', 'projdv7.3_ba-10p3', 'projdv7.3_ba-10p4', 'projdv7.3_ba-10p5', 'projdv7.3_ba-20p1', 'projdv7.3_ba-20p2', 'projdv7.3_ba-20p3', 'projdv7.3_ba-20p4', 'projdv7.3_ba-20p5']

===== Processing MTD =====
  ODA read: 5.0 s

===== Collecting WDA regridded files for MTD =====
WDA files found: 14
  WDA read: 0.2 s


In [49]:
results['MTD'].WDA_ds_rgd

<xarray.Dataset> Size: 8MB
Dimensions:  (time: 792, lat: 22, lon: 60)
Coordinates:
    z_t      (time, lat, lon) float32 4MB dask.array<chunksize=(12, 22, 60), meta=np.ndarray>
  * time     (time) object 6kB 1955-01-17 00:00:00 ... 2020-12-17 00:00:00
  * lat      (lat) float64 176B 19.0 20.0 21.0 22.0 23.0 ... 37.0 38.0 39.0 40.0
  * lon      (lon) float64 480B 120.0 121.0 122.0 123.0 ... 177.0 178.0 179.0
Data variables:
    MTD      (time, lat, lon) float32 4MB dask.array<chunksize=(12, 22, 60), meta=np.ndarray>

In [31]:
results["nDIC"] = CESM2_NWP_config()

for key in ["ODA", "LE", "WDA"]:
# for key in ["ODA"]:

    dic_obj  = getattr(results["DIC"],  f"{key}_ds_rgd")
    salt_obj = getattr(results["SALT"], f"{key}_ds_rgd")

    # get DataArrays
    dic_da  = dic_obj["DIC"]  if isinstance(dic_obj,  xr.Dataset) else dic_obj
    salt_da = salt_obj["SALT"] if isinstance(salt_obj, xr.Dataset) else salt_obj

    dic_da, salt_da = xr.align(dic_da, salt_da, join="inner")

    ndic_da = dic_da * 35.0 / salt_da
    ndic_da.name = "nDIC"
    ndic_da.attrs["long_name"] = "Salinity-normalized DIC (S=35)"
    ndic_da.attrs["units"] = dic_da.attrs.get("units","")

    setattr(results["nDIC"], f"{key}_ds_rgd", ndic_da.to_dataset())

In [49]:
# # ============================================================
# # Save nDIC inventories (MLD reference)
# # upper: z <= MLD
# # lower: z >  MLD
# # total: full column
# # same chunk structure as MLD files
# # ============================================================

# import xarray as xr
# import numpy as np
# import glob
# import os

# # ------------------------------------------------------------
# # INPUT
# # ------------------------------------------------------------
# ndic = results["nDIC"].ODA_ds_rgd["nDIC"]   # (ens_ODA,time,z_t,lat,lon)
# dz   = ds_grid.dz                           # (z_t) in cm

# # --- z_t coordinate 정확히 dz 기준으로 통일
# ndic = ndic.assign_coords(z_t=dz.z_t)

# # --- dz → meters (inventory 단위 mmol m-2 맞추기)
# dz_m = dz / 100.0

# MLD_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA"
# OUT_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA"

# ens_dim = "ens_ODA"

# # ------------------------------------------------------------
# # helpers
# # ------------------------------------------------------------
# def read_MLD_filelist(ens):
#     ens_dir = os.path.join(MLD_ROOT, f"{ens:02d}")
#     return sorted(glob.glob(os.path.join(ens_dir, "MLD_*.nc")))


# def compute_inventory_MLD(nd, MLD, dz_1d_m):
#     """
#     nd:  (time,z_t,lat,lon)
#     MLD: (time,lat,lon)  in cm
#     dz_1d_m: (z_t) in meters
#     """

#     # --- align time/lat/lon
#     nd, MLD = xr.align(nd, MLD, join="inner")

#     # --- broadcast z
#     z3 = nd.z_t.broadcast_like(nd)

#     # --- broadcast dz
#     dz3 = dz_1d_m.broadcast_like(nd)

#     # --- masks
#     upper_mask = z3 <= MLD
#     lower_mask = z3 >  MLD

#     # --- integrate
#     upper = (nd.where(upper_mask) * dz3).sum("z_t")
#     lower = (nd.where(lower_mask) * dz3).sum("z_t")
#     total = upper + lower

#     return upper, lower, total


# # ------------------------------------------------------------
# # loop ensembles
# # ------------------------------------------------------------
# for ens in ndic[ens_dim].values:

#     print("==== ens", ens, "====")

#     nd_full = ndic.sel({ens_dim: ens})

#     mld_files = read_MLD_filelist(int(ens))
#     out_dir = os.path.join(OUT_ROOT, f"{int(ens):02d}")
#     os.makedirs(out_dir, exist_ok=True)

#     for mld_file in mld_files:

#         print("MLD:", mld_file)

#         ds_mld = xr.open_dataset(mld_file)
#         MLD = ds_mld["MLD"]   # (time,lat,lon) cm

#         t0 = MLD.time.values[0]
#         t1 = MLD.time.values[-1]

#         nd_chunk = nd_full.sel(time=slice(t0, t1))

#         # --- inventory
#         upper, lower, total = compute_inventory_MLD(
#             nd_chunk,
#             MLD,
#             dz_m
#         )

#         ds_out = xr.Dataset({
#             "nDIC_upper_inventory": upper,
#             "nDIC_lower_inventory": lower,
#             "nDIC_total_inventory": total
#         })

#         for v in ds_out.data_vars:
#             ds_out[v].attrs["units"] = "mmol m-2"
#             ds_out[v].attrs["reference"] = "MLD"

#         fname = os.path.basename(mld_file).replace("MLD", "nDIC_INV")
#         out_file = os.path.join(out_dir, fname)

#         encoding = {
#             v: {"zlib": True, "complevel": 4, "dtype": "float32"}
#             for v in ds_out.data_vars
#         }

#         ds_out.to_netcdf(out_file, encoding=encoding)

#         ds_mld.close()
#         ds_out.close()

#         print("OUT:", out_file)


==== ens 10 ====
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1955.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA/10/nDIC_INV_ODA_10_1955.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1956.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA/10/nDIC_INV_ODA_10_1956.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1957.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA/10/nDIC_INV_ODA_10_1957.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1958.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA/10/nDIC_INV_ODA_10_1958.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1959.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA/10/nDIC_INV_ODA_10_1959.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1960.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA/10/nDIC_INV_ODA_10_1960.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_OD

In [37]:
# ============================================================
# Save nDIC inventories (MLD reference)
# ODA / LE / WDA
# - 0-based ensemble index (00,01,02,…)
# - Toggle experiments at top
# - Skip existing files
# - Print only one example file per ensemble
# ============================================================

import xarray as xr
import numpy as np
import glob
import os

# ------------------------------------------------------------
# USER TOGGLE
# ------------------------------------------------------------
RUN_EXP = {
    "ODA": False,
    "LE":  False,
    "WDA": True
}

# ------------------------------------------------------------
# GRID
# ------------------------------------------------------------
dz   = ds_grid.dz
dz_m = dz / 100.0  # cm → m

# ------------------------------------------------------------
# EXPERIMENT CONFIG
# ------------------------------------------------------------
EXP_CFG = {
    "ODA": {
        "ds": results["nDIC"].ODA_ds_rgd["nDIC"],
        "ens_dim": "ens_ODA",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/ODA",
        "has_ens": True,
        "ens_is_numeric": True
    },
    "LE": {
        "ds": results["nDIC"].LE_ds_rgd["nDIC"],
        "ens_dim": "ens_LE",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/LE",
        "has_ens": True,
        "ens_is_numeric": False
    },
    "WDA": {
        "ds": results["nDIC"].WDA_ds_rgd["nDIC"],
        "ens_dim": None,
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/WDA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/WDA",
        "has_ens": False,
        "ens_is_numeric": False
    }
}

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def read_MLD_filelist(root, ens_idx=None):
    """Return sorted list of MLD files.
    ens_idx is 0-based → folder 00,01,02,...
    """
    if ens_idx is None:
        return sorted(glob.glob(os.path.join(root, "MLD_*.nc")))
    ens_dir = os.path.join(root, f"{ens_idx:02d}")
    return sorted(glob.glob(os.path.join(ens_dir, "MLD_*.nc")))


def compute_inventory_MLD(nd, MLD, dz_1d_m):
    """Compute upper/lower/total nDIC inventory relative to MLD."""
    nd, MLD = xr.align(nd, MLD, join="inner")

    z3  = nd.z_t.broadcast_like(nd)
    dz3 = dz_1d_m.broadcast_like(nd)

    upper = (nd.where(z3 <= MLD) * dz3).sum("z_t")
    lower = (nd.where(z3 >  MLD) * dz3).sum("z_t")
    total = upper + lower

    return upper, lower, total


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
for EXP, cfg in EXP_CFG.items():

    if not RUN_EXP.get(EXP, False):
        continue

    print(f"\n[{EXP}]")

    ndic = cfg["ds"].assign_coords(z_t=dz.z_t)
    OUT_ROOT = cfg["OUT_ROOT"]
    MLD_ROOT = cfg["MLD_ROOT"]

    # --------------------------------------------------------
    # ENSEMBLE EXPERIMENTS (ODA / LE)
    # --------------------------------------------------------
    if cfg["has_ens"]:
        ens_dim = cfg["ens_dim"]

        for i, ens in enumerate(ndic[ens_dim].values):

            # 0-based ensemble index
            if cfg["ens_is_numeric"]:
                ens_idx = int(ens)
            else:
                ens_idx = i

            nd_full = ndic.sel({ens_dim: ens})
            mld_files = read_MLD_filelist(MLD_ROOT, ens_idx)

            out_dir = os.path.join(OUT_ROOT, f"{ens_idx:02d}")
            os.makedirs(out_dir, exist_ok=True)

            printed = False

            for mld_file in mld_files:

                fname = os.path.basename(mld_file).replace("MLD", "nDIC_INV")
                out_file = os.path.join(out_dir, fname)

                if os.path.exists(out_file):
                    continue

                ds_mld = xr.open_dataset(mld_file)
                MLD = ds_mld["MLD"]

                t0 = MLD.time.values[0]
                t1 = MLD.time.values[-1]

                nd_chunk = nd_full.sel(time=slice(t0, t1))

                upper, lower, total = compute_inventory_MLD(
                    nd_chunk, MLD, dz_m
                )

                ds_out = xr.Dataset({
                    "nDIC_upper_inventory": upper,
                    "nDIC_lower_inventory": lower,
                    "nDIC_total_inventory": total
                })

                for v in ds_out.data_vars:
                    ds_out[v].attrs["units"] = "mmol m-2"
                    ds_out[v].attrs["reference"] = "MLD"

                encoding = {
                    v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                    for v in ds_out.data_vars
                }

                ds_out.to_netcdf(out_file, encoding=encoding)

                ds_mld.close()
                ds_out.close()

                if not printed:
                    print(f" ens {ens_idx:02d}: {out_file}")
                    printed = True

    # --------------------------------------------------------
    # SINGLE RUN (WDA)
    # --------------------------------------------------------
    else:
        out_dir = OUT_ROOT
        os.makedirs(out_dir, exist_ok=True)

        printed = False
        mld_files = read_MLD_filelist(MLD_ROOT, None)

        for mld_file in mld_files:

            fname = os.path.basename(mld_file).replace("MLD", "nDIC_INV")
            out_file = os.path.join(out_dir, fname)

            if os.path.exists(out_file):
                continue

            ds_mld = xr.open_dataset(mld_file)
            MLD = ds_mld["MLD"]

            t0 = MLD.time.values[0]
            t1 = MLD.time.values[-1]

            nd_chunk = ndic.sel(time=slice(t0, t1))

            upper, lower, total = compute_inventory_MLD(
                nd_chunk, MLD, dz_m
            )

            ds_out = xr.Dataset({
                "nDIC_upper_inventory": upper,
                "nDIC_lower_inventory": lower,
                "nDIC_total_inventory": total
            })

            for v in ds_out.data_vars:
                ds_out[v].attrs["units"] = "mmol m-2"
                ds_out[v].attrs["reference"] = "MLD"

            encoding = {
                v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                for v in ds_out.data_vars
            }

            ds_out.to_netcdf(out_file, encoding=encoding)

            ds_mld.close()
            ds_out.close()

            if not printed:
                print(f" single: {out_file}")
                printed = True


[WDA]
 single: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV/WDA/nDIC_INV_WDA_1955.nc


In [54]:
# ============================================================
# Save nDIC inventories (between MLD and MTD)
# ODA / LE / WDA
# - integrate:  MLD < z <= MTD
# - MLD: read from files (as before)
# - MTD: use variable already in memory (same structure as nDIC)
# - 0-based ensemble index (00,01,02,…)
# - Skip existing files
# - Print only one example file per ensemble
# ============================================================

import xarray as xr
import numpy as np
import glob
import os

# ------------------------------------------------------------
# USER TOGGLE
# ------------------------------------------------------------
RUN_EXP = {
    "ODA": True,
    "LE":  False,
    "WDA": False
}

# ------------------------------------------------------------
# GRID
# ------------------------------------------------------------
dz   = ds_grid.dz
dz_m = dz / 100.0  # cm → m

# ------------------------------------------------------------
# EXPERIMENT CONFIG
# ------------------------------------------------------------
EXP_CFG = {
    "ODA": {
        "ndic": results["nDIC"].ODA_ds_rgd["nDIC"],
        "mtd":  results["MTD"].ODA_ds_rgd["MTD"],
        "ens_dim": "ens_ODA",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA",
        "has_ens": True,
        "ens_is_numeric": True
    },
    "LE": {
        "ndic": results["nDIC"].LE_ds_rgd["nDIC"],
        # "mtd":  results["MTD"].LE_ds_rgd["MTD"],
        "ens_dim": "ens_LE",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/LE",
        "has_ens": True,
        "ens_is_numeric": False
    },
    "WDA": {
        "ndic": results["nDIC"].WDA_ds_rgd["nDIC"],
        "mtd":  results["MTD"].WDA_ds_rgd["MTD"],
        "ens_dim": None,
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/WDA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/WDA",
        "has_ens": False,
        "ens_is_numeric": False
    }
}

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def read_MLD_filelist(root, ens_idx=None):
    """Return sorted list of MLD files.
    ens_idx is 0-based → folder 00,01,02,...
    """
    if ens_idx is None:
        return sorted(glob.glob(os.path.join(root, "MLD_*.nc")))
    ens_dir = os.path.join(root, f"{ens_idx:02d}")
    return sorted(glob.glob(os.path.join(ens_dir, "MLD_*.nc")))


def compute_inventory_between(nd, MLD, MTD, dz_1d_m):
    """Integrate nDIC between MLD and MTD (MLD < z <= MTD)."""
    nd, MLD, MTD = xr.align(nd, MLD, MTD, join="inner")

    z3  = nd.z_t.broadcast_like(nd)
    dz3 = dz_1d_m.broadcast_like(nd)

    mask = (z3 > MLD) & (z3 <= MTD)
    mid = (nd.where(mask) * dz3).sum("z_t")

    return mid


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
for EXP, cfg in EXP_CFG.items():

    if not RUN_EXP.get(EXP, False):
        continue

    print(f"\n[{EXP}]")

    ndic = cfg["ndic"].assign_coords(z_t=dz.z_t)
    mtd  = cfg["mtd"]   # MTD is 2D depth field, no z_t dimension

    OUT_ROOT = cfg["OUT_ROOT"]
    MLD_ROOT = cfg["MLD_ROOT"]

    # --------------------------------------------------------
    # ENSEMBLE EXPERIMENTS (ODA / LE)
    # --------------------------------------------------------
    if cfg["has_ens"]:

        ens_dim = cfg["ens_dim"]

        for i, ens in enumerate(ndic[ens_dim].values):

            # 0-based ensemble index
            ens_idx = int(ens) if cfg["ens_is_numeric"] else i

            nd_full  = ndic.sel({ens_dim: ens})
            mtd_full = mtd.sel({ens_dim: ens})

            mld_files = read_MLD_filelist(MLD_ROOT, ens_idx)

            out_dir = os.path.join(OUT_ROOT, f"{ens_idx:02d}")
            os.makedirs(out_dir, exist_ok=True)

            printed = False

            for mld_file in mld_files:

                fname = os.path.basename(mld_file).replace("MLD", "nDIC_INV_MLD_MTD")
                out_file = os.path.join(out_dir, fname)

                if os.path.exists(out_file):
                    continue

                ds_mld = xr.open_dataset(mld_file)
                MLD = ds_mld["MLD"]

                t0 = MLD.time.values[0]
                t1 = MLD.time.values[-1]

                nd_chunk  = nd_full.sel(time=slice(t0, t1))
                mtd_chunk = mtd_full.sel(time=slice(t0, t1))

                mid = compute_inventory_between(nd_chunk, MLD, mtd_chunk, dz_m)

                ds_out = xr.Dataset({
                    "nDIC_MLD_MTD_inventory": mid
                })

                ds_out["nDIC_MLD_MTD_inventory"].attrs["units"] = "mmol m-2"
                ds_out["nDIC_MLD_MTD_inventory"].attrs["reference"] = "MLD < z <= MTD"

                encoding = {
                    "nDIC_MLD_MTD_inventory": {
                        "zlib": True, "complevel": 4, "dtype": "float32"
                    }
                }

                ds_out.to_netcdf(out_file, encoding=encoding)

                ds_mld.close()
                ds_out.close()

                if not printed:
                    print(f" ens {ens_idx:02d}: {out_file}")
                    printed = True

    # --------------------------------------------------------
    # SINGLE RUN (WDA)
    # --------------------------------------------------------
    else:

        out_dir = OUT_ROOT
        os.makedirs(out_dir, exist_ok=True)

        printed = False
        mld_files = read_MLD_filelist(MLD_ROOT, None)

        for mld_file in mld_files:

            fname = os.path.basename(mld_file).replace("MLD", "nDIC_INV_MLD_MTD")
            out_file = os.path.join(out_dir, fname)

            if os.path.exists(out_file):
                continue

            ds_mld = xr.open_dataset(mld_file)
            MLD = ds_mld["MLD"]

            t0 = MLD.time.values[0]
            t1 = MLD.time.values[-1]

            nd_chunk  = ndic.sel(time=slice(t0, t1))
            mtd_chunk = mtd.sel(time=slice(t0, t1))

            mid = compute_inventory_between(nd_chunk, MLD, mtd_chunk, dz_m)

            ds_out = xr.Dataset({
                "nDIC_MLD_MTD_inventory": mid
            })

            ds_out["nDIC_MLD_MTD_inventory"].attrs["units"] = "mmol m-2"
            ds_out["nDIC_MLD_MTD_inventory"].attrs["reference"] = "MLD < z <= MTD"

            encoding = {
                "nDIC_MLD_MTD_inventory": {
                    "zlib": True, "complevel": 4, "dtype": "float32"
                }
            }

            ds_out.to_netcdf(out_file, encoding=encoding)

            ds_mld.close()
            ds_out.close()

            if not printed:
                print(f" single: {out_file}")
                printed = True


[ODA]
 ens 10: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/10/nDIC_INV_MLD_MTD_ODA_10_1955.nc
 ens 11: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/11/nDIC_INV_MLD_MTD_ODA_11_1955.nc
 ens 12: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/12/nDIC_INV_MLD_MTD_ODA_12_1955.nc
 ens 13: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/13/nDIC_INV_MLD_MTD_ODA_13_1955.nc
 ens 14: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/14/nDIC_INV_MLD_MTD_ODA_14_1955.nc
 ens 15: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/15/nDIC_INV_MLD_MTD_ODA_15_1955.nc
 ens 16: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/16/nDIC_INV_MLD_MTD_ODA_16_1955.nc
 ens 17: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/17/nDIC_INV_MLD_MTD_ODA_17_1955.nc
 ens 18: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/18/nDIC_INV_MLD_MTD_ODA_18_1955.nc
 ens 19: /mnt/lustre/proj/kimyy/tr_sysong/fld/nDIC_INV_MLD_MTD/ODA/19/nDIC_

In [47]:
# # ============================================================
# # Save DIC inventories (MLD reference, unit-correct)
# # ============================================================

# import xarray as xr
# import numpy as np
# import glob
# import os

# dic = results["DIC"].ODA_ds_rgd["DIC"]
# dz  = ds_grid.dz

# MLD_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA"
# OUT_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA"

# ens_dim = "ens_ODA"

# # ---- unify vertical grid
# dic = dic.assign_coords(z_t=dz.z_t)

# # ---- dz in meters
# dz_m = dz / 100.0

# def read_MLD_filelist(ens):
#     ens_dir = os.path.join(MLD_ROOT, f"{ens:02d}")
#     return sorted(glob.glob(os.path.join(ens_dir, "MLD_*.nc")))


# def compute_inventory(dc, MLD):

#     dc, MLD = xr.align(dc, MLD, join="inner")

#     dz3 = dz_m.broadcast_like(dc)
#     z3  = dc.z_t.broadcast_like(dc)

#     upper_mask = z3 <= MLD
#     lower_mask = z3 >  MLD

#     upper = (dc.where(upper_mask) * dz3).sum("z_t")
#     lower = (dc.where(lower_mask) * dz3).sum("z_t")
#     total = upper + lower

#     return upper, lower, total


# for ens in dic[ens_dim].values:

#     print("==== ens", ens, "====")

#     dc_full = dic.sel({ens_dim: ens})
#     mld_files = read_MLD_filelist(ens)

#     out_dir = os.path.join(OUT_ROOT, f"{ens:02d}")
#     os.makedirs(out_dir, exist_ok=True)

#     for mld_file in mld_files:

#         print("MLD:", mld_file)

#         ds_mld = xr.open_dataset(mld_file)
#         MLD = ds_mld["MLD"]   # cm

#         t0 = MLD.time.values[0]
#         t1 = MLD.time.values[-1]
#         dc_chunk = dc_full.sel(time=slice(t0, t1))

#         upper, lower, total = compute_inventory(dc_chunk, MLD)

#         ds_out = xr.Dataset({
#             "DIC_upper_inventory": upper,
#             "DIC_lower_inventory": lower,
#             "DIC_total_inventory": total
#         })

#         for v in ds_out.data_vars:
#             ds_out[v].attrs["units"] = "mmol m-2"
#             ds_out[v].attrs["long_name"] = v.replace("_", " ")

#         fname = os.path.basename(mld_file).replace("MLD", "DIC_INV")
#         out_file = os.path.join(out_dir, fname)

#         encoding = {
#             v: {"zlib": True, "complevel": 4, "dtype": "float32"}
#             for v in ds_out.data_vars
#         }

#         ds_out.to_netcdf(out_file, encoding=encoding)

#         ds_mld.close()
#         ds_out.close()

#         print("OUT:", out_file)


==== ens 10 ====
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1955.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA/10/DIC_INV_ODA_10_1955.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1956.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA/10/DIC_INV_ODA_10_1956.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1957.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA/10/DIC_INV_ODA_10_1957.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1958.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA/10/DIC_INV_ODA_10_1958.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1959.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA/10/DIC_INV_ODA_10_1959.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1960.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA/10/DIC_INV_ODA_10_1960.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1961.nc

In [38]:
# ============================================================
# Save DIC inventories (MLD reference)
# ODA / LE / WDA
# - 0-based ensemble index (00,01,02,…)
# - Toggle experiments at top
# - Skip existing files
# - Print only one example file per ensemble
# ============================================================

import xarray as xr
import numpy as np
import glob
import os

# ------------------------------------------------------------
# USER TOGGLE
# ------------------------------------------------------------
RUN_EXP = {
    "ODA": False,
    "LE":  False,
    "WDA": True
}

# ------------------------------------------------------------
# GRID
# ------------------------------------------------------------
dz   = ds_grid.dz
dz_m = dz / 100.0  # cm → m

# ------------------------------------------------------------
# EXPERIMENT CONFIG
# ------------------------------------------------------------
EXP_CFG = {
    "ODA": {
        "ds": results["DIC"].ODA_ds_rgd["DIC"],
        "ens_dim": "ens_ODA",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/ODA",
        "has_ens": True,
        "ens_is_numeric": True
    },
    "LE": {
        "ds": results["DIC"].LE_ds_rgd["DIC"],
        "ens_dim": "ens_LE",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/LE",
        "has_ens": True,
        "ens_is_numeric": False
    },
    "WDA": {
        "ds": results["DIC"].WDA_ds_rgd["DIC"],
        "ens_dim": None,
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/WDA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/WDA",
        "has_ens": False,
        "ens_is_numeric": False
    }
}

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def read_MLD_filelist(root, ens_idx=None):
    if ens_idx is None:
        return sorted(glob.glob(os.path.join(root, "MLD_*.nc")))
    ens_dir = os.path.join(root, f"{ens_idx:02d}")
    return sorted(glob.glob(os.path.join(ens_dir, "MLD_*.nc")))


def compute_inventory_MLD(dc, MLD, dz_1d_m):
    dc, MLD = xr.align(dc, MLD, join="inner")

    z3  = dc.z_t.broadcast_like(dc)
    dz3 = dz_1d_m.broadcast_like(dc)

    upper = (dc.where(z3 <= MLD) * dz3).sum("z_t")
    lower = (dc.where(z3 >  MLD) * dz3).sum("z_t")
    total = upper + lower

    return upper, lower, total


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
for EXP, cfg in EXP_CFG.items():

    if not RUN_EXP.get(EXP, False):
        continue

    print(f"\n[{EXP}]")

    dic = cfg["ds"].assign_coords(z_t=dz.z_t)
    OUT_ROOT = cfg["OUT_ROOT"]
    MLD_ROOT = cfg["MLD_ROOT"]

    # --------------------------------------------------------
    # ENSEMBLE EXPERIMENTS (ODA / LE)
    # --------------------------------------------------------
    if cfg["has_ens"]:
        ens_dim = cfg["ens_dim"]

        for i, ens in enumerate(dic[ens_dim].values):

            if cfg["ens_is_numeric"]:
                ens_idx = int(ens)
            else:
                ens_idx = i

            dc_full = dic.sel({ens_dim: ens})
            mld_files = read_MLD_filelist(MLD_ROOT, ens_idx)

            out_dir = os.path.join(OUT_ROOT, f"{ens_idx:02d}")
            os.makedirs(out_dir, exist_ok=True)

            printed = False

            for mld_file in mld_files:

                fname = os.path.basename(mld_file).replace("MLD", "DIC_INV")
                out_file = os.path.join(out_dir, fname)

                if os.path.exists(out_file):
                    continue

                ds_mld = xr.open_dataset(mld_file)
                MLD = ds_mld["MLD"]

                t0 = MLD.time.values[0]
                t1 = MLD.time.values[-1]
                dc_chunk = dc_full.sel(time=slice(t0, t1))

                upper, lower, total = compute_inventory_MLD(
                    dc_chunk, MLD, dz_m
                )

                ds_out = xr.Dataset({
                    "DIC_upper_inventory": upper,
                    "DIC_lower_inventory": lower,
                    "DIC_total_inventory": total
                })

                for v in ds_out.data_vars:
                    ds_out[v].attrs["units"] = "mmol m-2"
                    ds_out[v].attrs["long_name"] = v.replace("_", " ")

                encoding = {
                    v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                    for v in ds_out.data_vars
                }

                ds_out.to_netcdf(out_file, encoding=encoding)

                ds_mld.close()
                ds_out.close()

                if not printed:
                    print(f" ens {ens_idx:02d}: {out_file}")
                    printed = True

    # --------------------------------------------------------
    # SINGLE RUN (WDA)
    # --------------------------------------------------------
    else:
        out_dir = OUT_ROOT
        os.makedirs(out_dir, exist_ok=True)

        printed = False
        mld_files = read_MLD_filelist(MLD_ROOT, None)

        for mld_file in mld_files:

            fname = os.path.basename(mld_file).replace("MLD", "DIC_INV")
            out_file = os.path.join(out_dir, fname)

            if os.path.exists(out_file):
                continue

            ds_mld = xr.open_dataset(mld_file)
            MLD = ds_mld["MLD"]

            t0 = MLD.time.values[0]
            t1 = MLD.time.values[-1]
            dc_chunk = dic.sel(time=slice(t0, t1))

            upper, lower, total = compute_inventory_MLD(
                dc_chunk, MLD, dz_m
            )

            ds_out = xr.Dataset({
                "DIC_upper_inventory": upper,
                "DIC_lower_inventory": lower,
                "DIC_total_inventory": total
            })

            for v in ds_out.data_vars:
                ds_out[v].attrs["units"] = "mmol m-2"
                ds_out[v].attrs["long_name"] = v.replace("_", " ")

            encoding = {
                v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                for v in ds_out.data_vars
            }

            ds_out.to_netcdf(out_file, encoding=encoding)

            ds_mld.close()
            ds_out.close()

            if not printed:
                print(f" single: {out_file}")
                printed = True


[WDA]
 single: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV/WDA/DIC_INV_WDA_1955.nc


In [48]:
# ============================================================
# Save DIC inventories (300 m reference)
# upper: z <= 300 m
# lower: z > 300 m
# total: full column
# same chunk structure as MLD files
# ============================================================

import xarray as xr
import numpy as np
import glob
import os

# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------
dic = results["DIC"].ODA_ds_rgd["DIC"]   # (ens_ODA,time,z_t,lat,lon)
dz  = ds_grid.dz                         # (z_t)  in cm

# --- z_t coordinate 정확히 dz 기준으로 통일
dic = dic.assign_coords(z_t=dz.z_t)

# --- dz → meters (inventory 단위 mmol m-2 맞추기)
dz_m = dz / 100.0

MLD_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA"
OUT_ROOT = "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA"

ens_dim = "ens_ODA"
ZREF = 30000.0   # cm (300 m)

# ------------------------------------------------------------
# helpers
# ------------------------------------------------------------
def read_MLD_filelist(ens):
    ens_dir = os.path.join(MLD_ROOT, f"{ens:02d}")
    return sorted(glob.glob(os.path.join(ens_dir, "MLD_*.nc")))


def compute_inventory_300m(dc, dz_1d_m, zref_cm=30000.0):
    """
    dc: (time,z_t,lat,lon)
    dz_1d_m: (z_t) in meters
    """

    # --- z indices
    z = dc.z_t.values
    upper_idx = np.where(z <= zref_cm)[0]
    lower_idx = np.where(z >  zref_cm)[0]

    # --- slice
    dc_upper = dc.isel(z_t=upper_idx)
    dc_lower = dc.isel(z_t=lower_idx)

    dz_upper = dz_1d_m.isel(z_t=upper_idx)
    dz_lower = dz_1d_m.isel(z_t=lower_idx)

    # --- broadcast dz → 4D
    dz_upper_4d = dz_upper.broadcast_like(dc_upper)
    dz_lower_4d = dz_lower.broadcast_like(dc_lower)

    # --- integrate
    upper = (dc_upper * dz_upper_4d).sum("z_t")
    lower = (dc_lower * dz_lower_4d).sum("z_t")

    total = upper + lower

    return upper, lower, total


# ------------------------------------------------------------
# loop ensembles
# ------------------------------------------------------------
for ens in dic[ens_dim].values:

    print("==== ens", ens, "====")

    dc_full = dic.sel({ens_dim: ens})  # (time,z_t,lat,lon)

    mld_files = read_MLD_filelist(int(ens))
    out_dir = os.path.join(OUT_ROOT, f"{int(ens):02d}")
    os.makedirs(out_dir, exist_ok=True)

    for mld_file in mld_files:

        print("MLD:", mld_file)

        ds_mld = xr.open_dataset(mld_file)
        MLD = ds_mld["MLD"]

        t0 = MLD.time.values[0]
        t1 = MLD.time.values[-1]

        dc_chunk = dc_full.sel(time=slice(t0, t1))

        # --- inventory
        upper, lower, total = compute_inventory_300m(
            dc_chunk,
            dz_m,
            ZREF
        )

        ds_out = xr.Dataset({
            "DIC_upper_300m": upper,
            "DIC_lower_300m": lower,
            "DIC_total_300m": total
        })

        for v in ds_out.data_vars:
            ds_out[v].attrs["units"] = "mmol m-2"
            ds_out[v].attrs["reference_depth"] = "300 m"

        fname = os.path.basename(mld_file).replace("MLD", "DIC_INV_300m")
        out_file = os.path.join(out_dir, fname)

        encoding = {
            v: {"zlib": True, "complevel": 4, "dtype": "float32"}
            for v in ds_out.data_vars
        }

        ds_out.to_netcdf(out_file, encoding=encoding)

        ds_mld.close()
        ds_out.close()

        print("OUT:", out_file)


==== ens 10 ====
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1955.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA/10/DIC_INV_300m_ODA_10_1955.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1956.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA/10/DIC_INV_300m_ODA_10_1956.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1957.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA/10/DIC_INV_300m_ODA_10_1957.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1958.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA/10/DIC_INV_300m_ODA_10_1958.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1959.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA/10/DIC_INV_300m_ODA_10_1959.nc
MLD: /mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA/10/MLD_ODA_10_1960.nc
OUT: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA/10/DIC_INV_300m_ODA_10_1960.nc
MLD: /mnt/l

In [39]:
# ============================================================
# Save DIC inventories (300 m reference)
# ODA / LE / WDA
# - 0-based ensemble index (00,01,02,…)
# - Toggle experiments at top
# - Skip existing files
# - Print only one example file per ensemble
# ============================================================

import xarray as xr
import numpy as np
import glob
import os

# ------------------------------------------------------------
# USER TOGGLE
# ------------------------------------------------------------
RUN_EXP = {
    "ODA": False,
    "LE":  False,
    "WDA": True
}

# ------------------------------------------------------------
# GRID
# ------------------------------------------------------------
dz   = ds_grid.dz
dz_m = dz / 100.0  # cm → m
ZREF = 30000.0     # cm (300 m)

# ------------------------------------------------------------
# EXPERIMENT CONFIG
# ------------------------------------------------------------
EXP_CFG = {
    "ODA": {
        "ds": results["DIC"].ODA_ds_rgd["DIC"],
        "ens_dim": "ens_ODA",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/ODA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/ODA",
        "has_ens": True,
        "ens_is_numeric": True
    },
    "LE": {
        "ds": results["DIC"].LE_ds_rgd["DIC"],
        "ens_dim": "ens_LE",
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/LE",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/LE",
        "has_ens": True,
        "ens_is_numeric": False
    },
    "WDA": {
        "ds": results["DIC"].WDA_ds_rgd["DIC"],
        "ens_dim": None,
        "MLD_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/MLD/WDA",
        "OUT_ROOT": "/mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/WDA",
        "has_ens": False,
        "ens_is_numeric": False
    }
}

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def read_MLD_filelist(root, ens_idx=None):
    if ens_idx is None:
        return sorted(glob.glob(os.path.join(root, "MLD_*.nc")))
    ens_dir = os.path.join(root, f"{ens_idx:02d}")
    return sorted(glob.glob(os.path.join(ens_dir, "MLD_*.nc")))


def compute_inventory_300m(dc, dz_1d_m, zref_cm):
    z = dc.z_t.values

    upper_idx = np.where(z <= zref_cm)[0]
    lower_idx = np.where(z >  zref_cm)[0]

    dc_upper = dc.isel(z_t=upper_idx)
    dc_lower = dc.isel(z_t=lower_idx)

    dz_upper = dz_1d_m.isel(z_t=upper_idx)
    dz_lower = dz_1d_m.isel(z_t=lower_idx)

    dz_upper_4d = dz_upper.broadcast_like(dc_upper)
    dz_lower_4d = dz_lower.broadcast_like(dc_lower)

    upper = (dc_upper * dz_upper_4d).sum("z_t")
    lower = (dc_lower * dz_lower_4d).sum("z_t")
    total = upper + lower

    return upper, lower, total


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
for EXP, cfg in EXP_CFG.items():

    if not RUN_EXP.get(EXP, False):
        continue

    print(f"\n[{EXP}]")

    dic = cfg["ds"].assign_coords(z_t=dz.z_t)
    OUT_ROOT = cfg["OUT_ROOT"]
    MLD_ROOT = cfg["MLD_ROOT"]

    # --------------------------------------------------------
    # ENSEMBLE EXPERIMENTS
    # --------------------------------------------------------
    if cfg["has_ens"]:
        ens_dim = cfg["ens_dim"]

        for i, ens in enumerate(dic[ens_dim].values):

            if cfg["ens_is_numeric"]:
                ens_idx = int(ens)
            else:
                ens_idx = i

            dc_full = dic.sel({ens_dim: ens})
            mld_files = read_MLD_filelist(MLD_ROOT, ens_idx)

            out_dir = os.path.join(OUT_ROOT, f"{ens_idx:02d}")
            os.makedirs(out_dir, exist_ok=True)

            printed = False

            for mld_file in mld_files:

                fname = os.path.basename(mld_file).replace("MLD", "DIC_INV_300m")
                out_file = os.path.join(out_dir, fname)

                if os.path.exists(out_file):
                    continue

                ds_mld = xr.open_dataset(mld_file)
                MLD = ds_mld["MLD"]

                t0 = MLD.time.values[0]
                t1 = MLD.time.values[-1]
                dc_chunk = dc_full.sel(time=slice(t0, t1))

                upper, lower, total = compute_inventory_300m(
                    dc_chunk, dz_m, ZREF
                )

                ds_out = xr.Dataset({
                    "DIC_upper_300m": upper,
                    "DIC_lower_300m": lower,
                    "DIC_total_300m": total
                })

                for v in ds_out.data_vars:
                    ds_out[v].attrs["units"] = "mmol m-2"
                    ds_out[v].attrs["reference_depth"] = "300 m"

                encoding = {
                    v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                    for v in ds_out.data_vars
                }

                ds_out.to_netcdf(out_file, encoding=encoding)

                ds_mld.close()
                ds_out.close()

                if not printed:
                    print(f" ens {ens_idx:02d}: {out_file}")
                    printed = True

    # --------------------------------------------------------
    # SINGLE RUN (WDA)
    # --------------------------------------------------------
    else:
        out_dir = OUT_ROOT
        os.makedirs(out_dir, exist_ok=True)

        printed = False
        mld_files = read_MLD_filelist(MLD_ROOT, None)

        for mld_file in mld_files:

            fname = os.path.basename(mld_file).replace("MLD", "DIC_INV_300m")
            out_file = os.path.join(out_dir, fname)

            if os.path.exists(out_file):
                continue

            ds_mld = xr.open_dataset(mld_file)
            MLD = ds_mld["MLD"]

            t0 = MLD.time.values[0]
            t1 = MLD.time.values[-1]
            dc_chunk = dic.sel(time=slice(t0, t1))

            upper, lower, total = compute_inventory_300m(
                dc_chunk, dz_m, ZREF
            )

            ds_out = xr.Dataset({
                "DIC_upper_300m": upper,
                "DIC_lower_300m": lower,
                "DIC_total_300m": total
            })

            for v in ds_out.data_vars:
                ds_out[v].attrs["units"] = "mmol m-2"
                ds_out[v].attrs["reference_depth"] = "300 m"

            encoding = {
                v: {"zlib": True, "complevel": 4, "dtype": "float32"}
                for v in ds_out.data_vars
            }

            ds_out.to_netcdf(out_file, encoding=encoding)

            ds_mld.close()
            ds_out.close()

            if not printed:
                print(f" single: {out_file}")
                printed = True


[WDA]
 single: /mnt/lustre/proj/kimyy/tr_sysong/fld/DIC_INV_300m/WDA/DIC_INV_300m_WDA_1955.nc
